## 1. Cài đặt thư viện

In [3]:
# Chỉ cài những cái bắt buộc
!pip install -q timm einops accelerate
!pip install -U datasets
!pip install transformers==4.47.0

## 2. Load Dataset (Tập test ảnh cho người khiếm thị)

In [4]:
from google.colab import drive, userdata
from datasets import load_dataset, load_from_disk
import os

# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn lưu dataset trên Drive
# Tôi đổi tên thư mục một chút để tránh nhầm lẫn với bản tải toàn bộ trước đó
drive_path = "/content/drive/MyDrive/HCMUS_Vietnamese_Dataset"

if os.path.exists(drive_path):
    print("🚀 Tìm thấy dataset Test trên Google Drive, đang load (chỉ mất vài giây)...")
    # Khi load từ disk đã lưu split='test', nó sẽ là một đối tượng Dataset trực tiếp
    dataset = load_from_disk(drive_path)

else:
    print("📥 Không tìm thấy trên Drive, đang tải DUY NHẤT tập TEST từ Hugging Face...")

    try:
        # Lấy token từ Secrets
        hf_token = userdata.get('HF_Ltro')

        # CHỈ TẢI TẬP TEST để tiết kiệm thời gian và dung lượng
        dataset = load_dataset(
            "pqthinh232/HCMUS-Vietnamese-Image-captioning-for-visually-impaired",
            data_files={"test": "test/*"},
            split="test",
            token=hf_token
        )

        print("💾 Đang lưu tập Test vào Google Drive để lần sau dùng luôn...")
        # Lưu đối tượng Dataset (split="test") vào ổ đĩa
        dataset.save_to_disk(drive_path)

    except Exception as e:
        print(f"❌ Lỗi: {e}")
        print("Hãy đảm bảo bạn đã tạo Secret tên 'HF_Ltro' và BẬT nút 'Notebook access' màu xanh.")

print(f"✅ Dataset đã sẵn sàng! Tổng số dòng dữ liệu (Test): {len(dataset)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Tìm thấy dataset Test trên Google Drive, đang load (chỉ mất vài giây)...
✅ Dataset đã sẵn sàng! Tổng số dòng dữ liệu (Test): 4000


## 3. Download Model (Vintern-1B-v3_5)

In [5]:
from huggingface_hub import snapshot_download
from google.colab import userdata
import os

# 1. Lấy token từ Secrets (Nhớ đặt tên trong bảng Secrets là HF_TOKEN nhé)
try:
    token = userdata.get('HF_Ltro')
except:
    token = None
    print("⚠️ Cảnh báo: Không tìm thấy Secret tên HF_TOKEN. Nếu model là public thì không sao.")

# 2. Tạo thư mục đích
target_dir = "pretrained/Vintern-1B-v3_5"
os.makedirs(target_dir, exist_ok=True)

print(f"🚀 Đang bắt đầu tải model về thư mục: {target_dir}")

# 3. Tải model bằng Python code (Thay cho huggingface-cli)
snapshot_download(
    repo_id="5CD-AI/Vintern-1B-v3_5",
    local_dir=target_dir,
    token=token,
    local_dir_use_symlinks=False,
    revision="main"
)

print("✅ Tải model HOÀN TẤT!")
# Liệt kê file để kiểm tra xem đã có chưa
print("Các file đã tải:")
print(os.listdir(target_dir))

🚀 Đang bắt đầu tải model về thư mục: pretrained/Vintern-1B-v3_5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

configuration_internvl_chat.py: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

modeling_intern_vit.py: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.75G [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

✅ Tải model HOÀN TẤT!
Các file đã tải:
['conversation.py', '.cache', 'vocab.json', 'configuration_intern_vit.py', 'configuration_internvl_chat.py', 'modeling_intern_vit.py', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'README.md', 'config.json', 'tokenizer_config.json', 'added_tokens.json', '.gitattributes', 'generation_config.json', 'modeling_internvl_chat.py']


## 4. Load Model

In [6]:
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
import matplotlib.pyplot as plt

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_aspect_ratio = find_closest_aspect_ratio(aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    resized_img = image.resize((target_width, target_height))
    processed_images =[]
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    if isinstance(image_file, str):
        image = Image.open(image_file).convert('RGB')
    else:
        image = image_file
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values =[transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

In [7]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = "/content/pretrained/Vintern-1B-v3_5"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
model = AutoModel.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    use_flash_attn=False,
).eval().cuda()


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


configuration_internvl_chat.py: 0.00B [00:00, ?B/s]

configuration_intern_vit.py: 0.00B [00:00, ?B/s]

...

conversation.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/5CD-AI/Vintern-1B-v3_5:
- conversation.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_intern_vit.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/5CD-AI/Vintern-1B-v3_5:
- modeling_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/5CD-AI/Vintern-1B-v3_5:
- modeling_internvl_chat.py
- conversation.py
- modeling_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


## 5. Test Inference

In [12]:
import os
import time
import json
import torch
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. ĐO THÔNG SỐ TĨNH (Params & Disk Size)
# ==========================================
model_path = "/content/pretrained/Vintern-1B-v3_5"

# Đo Disk Size (GB)
def get_dir_size(path):
    total_size = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size / (1024**3) # Đổi sang GB

disk_size_gb = get_dir_size(model_path)

# Đo Params (Triệu tham số)
total_params = sum(p.numel() for p in model.parameters())
params_m = total_params / 1e6

print(f"📊 [STATIC METRICS]")
print(f"- Params: {params_m:.2f} M (Triệu tham số)")
print(f"- Disk Size: {disk_size_gb:.2f} GB")
print("-" * 50)

# ==========================================
# 2. CẤU HÌNH PROMPT & INFERENCE
# ==========================================
UNIVERSAL_INFERENCE_PROMPT = "Hãy viết duy nhất một câu văn trôi chảy, tập trung thực hiện mô tả bối cảnh/vật cản, kèm theo lời khuyên cho người khiếm thị di chuyển an toàn ngắn gọn."

generation_config = dict(max_new_tokens=128, do_sample=False, num_beams=1, repetition_penalty=1.1, pad_token_id=151645)
question = f'<image>\n{UNIVERSAL_INFERENCE_PROMPT}'

# Chuẩn bị biến lưu kết quả
results_data = []
inference_times = []

# Reset bộ đo VRAM để lấy giá trị đỉnh (Peak VRAM) chính xác nhất
torch.cuda.reset_peak_memory_stats()

total_rows = len(dataset)
total_images = total_rows // 5

print(f"🚀 Bắt đầu Inference toàn bộ {total_images} ảnh ...")

# ==========================================
# 3. VÒNG LẶP CHẠY TOÀN BỘ ẢNH (800 ẢNH)
# ==========================================
# Dùng tqdm để hiển thị thanh tiến trình
for i in tqdm(range(0, total_rows, 5), desc="Processing Images"):
    sample = dataset[i]
    image = sample['image']

    # Lấy 5 câu mẫu (Reference) cho ảnh hiện tại
    ground_truths = [dataset[j]['caption'] for j in range(i, i + 5)]

    # Tiền xử lý ảnh (Không tính vào thời gian sinh chữ để đo Model thôi)
    pixel_values = load_image(image, max_num=6).to(torch.bfloat16).cuda()

    # --- BẮT ĐẦU ĐO THỜI GIAN INFERENCE ---
    start_time = time.time()

    with torch.no_grad(): # Tắt tính đạo hàm để tiết kiệm VRAM, chống tràn RAM
        response = model.chat(tokenizer, pixel_values, question, generation_config)

    end_time = time.time()
    # --- KẾT THÚC ĐO THỜI GIAN ---

    # Tính TẤT CẢ thời gian, kể cả 5 ảnh đầu
    inference_times.append(end_time - start_time)

    # Lưu dữ liệu của ảnh này vào danh sách
    results_data.append({
        "image_id": i // 5,
        "prediction": response,
        "references": ground_truths
    })

# ==========================================
# 4. ĐO THÔNG SỐ ĐỘNG (Time/Img & VRAM) VÀ LƯU FILE
# ==========================================
# Tính Time/Img trung bình (giây) trên TOÀN BỘ 800 ảnh
avg_time_per_img = np.mean(inference_times) if inference_times else 0

# Lấy Peak VRAM (GB)
peak_vram_gb = torch.cuda.max_memory_allocated() / (1024**3)

# Đóng gói toàn bộ dữ liệu để lưu
final_output = {
    "system_metrics": {
        "params_M": round(params_m, 2),
        "disk_size_GB": round(disk_size_gb, 2),
        "time_per_img_sec": round(avg_time_per_img, 4),
        "peak_vram_GB": round(peak_vram_gb, 2)
    },
    "predictions": results_data
}

# Lưu ra file JSON
save_path = '/content/drive/MyDrive/vintern_inference_results.json'
with open(save_path, 'w', encoding='utf-8') as f:
    json.dump(final_output, f, ensure_ascii=False, indent=4)

print("\n" + "=" * 50)
print(f"✅ ĐÃ HOÀN THÀNH INFERENCE {total_images} ẢNH!")
print(f"📊 [DYNAMIC METRICS]")
print(f"- Time/Img (Latency): {avg_time_per_img:.3f} giây/ảnh")
print(f"- Peak VRAM: {peak_vram_gb:.2f} GB")
print(f"💾 Toàn bộ kết quả (Metrics + Caption) đã được lưu tại: {save_path}")
print("=" * 50)

📊 [STATIC METRICS]
- Params: 938.19 M (Triệu tham số)
- Disk Size: 3.50 GB
--------------------------------------------------
🚀 Bắt đầu Inference toàn bộ 800 ảnh ...


Processing Images: 100%|██████████| 800/800 [48:10<00:00,  3.61s/it]


✅ ĐÃ HOÀN THÀNH INFERENCE 800 ẢNH!
📊 [DYNAMIC METRICS]
- Time/Img (Latency): 3.565 giây/ảnh
- Peak VRAM: 2.38 GB
💾 Toàn bộ kết quả (Metrics + Caption) đã được lưu tại: /content/drive/MyDrive/vintern_inference_results.json


In [14]:
!pip install -q pyvi
!pip install -q git+https://github.com/salaniz/pycocoevalcap.git
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [18]:
import json
import numpy as np
from pyvi import ViTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge

# ==========================================
# 1. LOAD DỮ LIỆU TỪ FILE JSON TRÊN DRIVE
# ==========================================
file_path = '/content/drive/MyDrive/vintern_inference_results.json'

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

predictions_data = data['predictions']
system_metrics = data['system_metrics']

# ==========================================
# 2. TIỀN XỬ LÝ: TÁCH TỪ TIẾNG VIỆT (WORD SEGMENT)
# ==========================================
print("✂️ Đang thực hiện Tách từ (Word Segmentation) Tiếng Việt...")
gts = {} # Ground Truths
res = {} # Results (Predictions)

for item in predictions_data:
    img_id = str(item['image_id'])

    # Tách từ cho Prediction (Sử dụng lowercase để chuẩn hóa)
    pred_seg = ViTokenizer.tokenize(item['prediction'].lower())
    res[img_id] = [pred_seg]

    # Tách từ cho 5 câu References
    refs_seg = [ViTokenizer.tokenize(ref.lower()) for ref in item['references']]
    gts[img_id] = refs_seg

# ==========================================
# 3. TÍNH TOÁN CÁC ĐỘ ĐO NGÔN NGỮ (TEXT METRICS)
# ==========================================
print("📈 Đang tính điểm BLEU, CIDEr, ROUGE-L...")

# Khởi tạo các hàm chấm điểm
scorers = [
    (Bleu(4), ["BLEU_1", "BLEU_2", "BLEU_3", "BLEU_4"]),
    (Cider(), "CIDEr"),
    (Rouge(), "ROUGE_L")
]

final_scores = {}

for scorer, method in scorers:
    # compute_score trả về (score_trung_binh, danh_sach_score_tung_anh)
    score, _ = scorer.compute_score(gts, res)

    if isinstance(method, list):
        for m, s in zip(method, score):
            final_scores[m] = round(s * 100, 2) # Chuyển sang thang điểm 100
    else:
        final_scores[method] = round(score * 100, 2)

# ==========================================
# 4. IN BÁO CÁO TỔNG HỢP (10 METRICS)
# ==========================================
print("\n" + "★" * 60)
print("🏆 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH VINTERN-1B-v3_5 🏆")
print("★" * 60)

print("\n🅰️ [CHẤT LƯỢNG NGÔN NGỮ - TEXT METRICS]")
print(f"   ➤ BLEU-1 : {final_scores['BLEU_1']}")
print(f"   ➤ BLEU-2 : {final_scores['BLEU_2']}")
print(f"   ➤ BLEU-3 : {final_scores['BLEU_3']}")
print(f"   ➤ BLEU-4 : {final_scores['BLEU_4']}")
print(f"   ➤ CIDEr  : {final_scores['CIDEr']}")
print(f"   ➤ ROUGE-L: {final_scores['ROUGE_L']}")

print("\n🅱️ [HIỆU NĂNG HỆ THỐNG - SYSTEM METRICS]")
print(f"   ➤ Tổng tham số (Params) : {system_metrics['params_M']} M")
print(f"   ➤ Dung lượng đĩa (Disk) : {system_metrics['disk_size_GB']} GB")
print(f"   ➤ Độ trễ (Time/Img)     : {system_metrics['time_per_img_sec']} giây/ảnh")
print(f"   ➤ VRAM đỉnh (Peak VRAM) : {system_metrics['peak_vram_GB']} GB")

print("\n" + "★" * 60)

# ==========================================
# 5. CẬP NHẬT KẾT QUẢ VÀO FILE JSON
# ==========================================
data['evaluation_results'] = final_scores
with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"💾 Bảng điểm đã được lưu đè vào file: {file_path}")

✂️ Đang thực hiện Tách từ (Word Segmentation) Tiếng Việt...
📈 Đang tính điểm BLEU, CIDEr, ROUGE-L...
{'testlen': 21703, 'reflen': 24369, 'guess': [21703, 20903, 20103, 19303], 'correct': [9157, 1673, 436, 107]}
ratio: 0.8905987114776606

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH VINTERN-1B-v3_5 🏆
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

🅰️ [CHẤT LƯỢNG NGÔN NGỮ - TEXT METRICS]
   ➤ BLEU-1 : 37.32
   ➤ BLEU-2 : 16.25
   ➤ BLEU-3 : 7.97
   ➤ BLEU-4 : 3.97
   ➤ CIDEr  : 6.11
   ➤ ROUGE-L: 17.38

🅱️ [HIỆU NĂNG HỆ THỐNG - SYSTEM METRICS]
   ➤ Tổng tham số (Params) : 938.19 M
   ➤ Dung lượng đĩa (Disk) : 3.5 GB
   ➤ Độ trễ (Time/Img)     : 3.5654 giây/ảnh
   ➤ VRAM đỉnh (Peak VRAM) : 2.38 GB

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
💾 Bảng điểm đã được lưu đè vào file: /content/drive/MyDrive/vintern_inference_results.json
